# Data Preparation
This version of the data preparation is before having access to the full text of the articles. Instead, we're running a test using just article titles.

In [ ]:
import pandas as pd
import numpy as np

## 0. Read and filter data

In [ ]:
raw_df = pd.read_csv('Data/posts-export-by-page-views-Feb-01-2025-Mar-05-2026-Masthead-Maine.csv')
raw_df.head()

In [ ]:
raw_df.shape

In [ ]:
raw_df.columns

In [ ]:
raw_df['Post id'].isna().sum()

In [ ]:
raw_df['Publish date'].isna().sum()

In [ ]:
i = 500
print(raw_df.URL[i],'\n',raw_df['Post id'][i])

I can either use 'URL' or 'Post id' as identifiable index. Can filter by either 'Post id' or 'Publish date'

In [ ]:
art_df = raw_df[~raw_df['Post id'].isna()].reset_index(drop=True) # full "raw" dataset that only contains articles
art_df.shape

In [ ]:
art_df['Publish date']

In [ ]:
# check for articles older than Feb 01 2025
# first, convert dates from str to datetime objects
import datetime
art_df['Publish_date'] = art_df['Publish date'].apply(lambda x : datetime.datetime.strptime(x, '%Y-%m-%d %H:%M'))

print((art_df.Publish_date > datetime.datetime.strptime('2025-01-31 23:59:00', '%Y-%m-%d %H:%M:%S')).sum(),' of ', art_df.shape[0],' articles within date range.')


In [ ]:
# filter them out
art_df = art_df[art_df.Publish_date > datetime.datetime.strptime('2025-01-31 23:59:00', '%Y-%m-%d %H:%M:%S')].reset_index(drop=True)
art_df.shape

In [ ]:
# load scraped text data
tagged_text = pd.read_csv('Data/text-tagged-articles.csv')
untagged_text = pd.read_csv('Data/text-untagged-articles.csv')
text_df = pd.concat([tagged_text,untagged_text],ignore_index=True)
print(text_df.shape)
text_df.head()

In [ ]:
art_df = art_df.merge(text_df, how='left', left_on='URL', right_on='url')
print(art_df.shape)
print(art_df.text[0])

## 1. 'User_Needs' column

In [ ]:
art_df['Tag_list'] = art_df.Tags.apply(lambda x : x.split(','))

user_tags = []
for i in range(len(art_df)):
    flag = 0
    for tag in art_df['Tag_list'][i]:
        if 'user_need' in tag:
            user_tags.append(tag[len('user_need: '):])
            flag = 1
    if flag:
        continue
    user_tags.append('none') # this is how they're labeled originally

In [ ]:
len(user_tags) # just checking that the len matches 

In [ ]:
art_df['User_Needs'] = user_tags
art_df.User_Needs.unique()

NOTE: 'other-not-news' tag is NOT part of the User Needs tags. **Should it be treated as 'none'?**
--> We'll remove them!

In [ ]:
art_df = art_df[(art_df.User_Needs != 'other-not-news')].reset_index(drop=True)
len(art_df)

## 2. Clean data for EDA

(copy-pasted the list of columns for reference)<br />
       'Apikey', 'URL', 'Title', 'Publish date', 'Authors', 'Section', 'Tags',<br />
       'Sort (Views)', 'Visitors', 'Views', 'Avg. views', 'Engaged minutes',<br />
       'Avg. minutes', 'New vis.', 'Views new vis.', 'Avg. views new vis.',<br />
       'Minutes New Vis.', 'Avg. minutes new vis.', 'Returning vis.',<br />
       'Views ret. vis.', 'Avg. views ret. vis.', 'Minutes Ret. Vis.',<br />
       'Avg. minutes ret. vis.', 'Desktop views', 'Mobile views',<br />
       'Tablet views', 'Search refs', 'Internal refs', 'Other refs',<br />
       'Direct refs', 'Social refs', 'Fb refs', 'Tw refs', 'Pi refs',<br />
       'Social interactions', 'Fb interactions', 'Pi interactions',<br />
       'Channel vis.', 'Website views', 'AMP views', 'Fb instant views',<br />
       'Post id', 'Views source', 'Views syndicated', 'Views by Site'<br />

In [ ]:
# choose columns to keep
columns = ['Apikey','URL','Title','text','Publish_date','Authors','Section','User_Needs', # metadata
           'Views','Avg. views','Engaged minutes','Avg. minutes', # some metrics (can be changed to include other metrics!)
           'Desktop views','Mobile views', 'Tablet views'] # device metrics
eda_df = art_df[columns]
eda_df.head()

In [ ]:
eda_df.count()

In [ ]:
eda_df['Tablet views'] = eda_df['Tablet views'].fillna(0)
eda_df.count()

In [ ]:
eda_df = eda_df.dropna(ignore_index=True)
eda_df.count()

In [ ]:
eda_df.to_csv('Data/EDA_data-FULL.csv', index=False) # save!

## 3. Clean data for ML

In [ ]:
# we're just cutting out unneeded columns and splitting into tagged/untagged datasets
valid_tags = ['update-me','give-me-perspective','educate-me',
              'connect-me','help-me','inspire-me']
ml_cols = ['URL','Title','text','User_Needs']

tagged_df = eda_df[ml_cols][eda_df.User_Needs.isin(valid_tags)]
print(tagged_df.shape[0],' tagged.')

untagged_df = eda_df[ml_cols][~eda_df.User_Needs.isin(valid_tags)]
print(untagged_df.shape[0],' untagged.')

In [ ]:
tagged_df.to_csv('Data/ML_tagged_data-FULL.csv',index=False)
untagged_df.to_csv('Data/ML_untagged_data-FULL.csv',index=False)